# LeWorldModel — Training on Colab T4

Everything runs **locally on the Colab instance** (no Drive mount).

**Pipeline :**
1. Check GPU
2. Clone repo
3. Generate simple pendulum dataset
4. Train LeWorldModel (JEPA + SIGReg)
5. Plot loss curves
6. Visualise latent space (PCA)
7. Download best checkpoint

**Architecture :** JEPA pur — encodeur online + target encoder EMA, prédiction dans l'espace latent,  
zéro supervision pixel. Le MLP de transition force ω (vitesse angulaire) dans z.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 3. Install dependencies

In [ ]:
!pip install -q Pillow matplotlib numpy
print('Dependencies ready.')

## 4. Generate dataset

Generates **2 000 trajectories × 500 frames** (64×64 px) of a simple pendulum.  
Each trajectory has random initial angle **and** random initial velocity.  
~5-10 min on Colab CPU.

During training, a random **window of 100 frames** is sampled from each 500-frame
trajectory at every epoch — giving 5× more temporal diversity than a fixed dataset.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from generate_dataset import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

### Quick dataset preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = np.load(f'{DATASET_DIR}/traj_0000.npz')
frames = sample['frames']   # (T, H, W, 3)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, np.linspace(0, len(frames)-1, 8, dtype=int)):
    ax.imshow(frames[idx])
    ax.axis('off')
    ax.set_title(f't={idx}', color='white', fontsize=8)
plt.suptitle('Trajectory 0 — 8 frames', color='white')
plt.tight_layout()
plt.show()

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `embed_dim` | 128 | latent dimension |
| `hidden_dim` | 512 | MLP transition FFN |
| `lam` | 0.5 | SIGReg weight — **aligné avec AE** |
| `ema_momentum` | 0.996 | target encoder EMA (τ) |
| `rollout_k` | 10 | horizon de prédiction — **aligné avec AE** (0.5 s = 10 × dt) |
| `mse_coef` | 0.1 | contrainte de magnitude dans la pred loss |
| `norm_coef` | 0.0 | désactivé — cause collapse avec rollout_k élevé |
| `seq_len` | 100 | window tirée aléatoirement dans 500 frames |
| `epochs` | 50 | |
| `batch_size` | 32 | fits T4 16 GB |
| `lr` | 1e-4 | AdamW + warmup 5ep + cosine |

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM    = 128
HIDDEN_DIM   = 512
LAM          = 0.5    # SIGReg weight — aligné avec AE
EMA_MOMENTUM = 0.996  # target encoder EMA momentum (τ)
ROLLOUT_K    = 10     # horizon de prédiction — aligné avec AE (0.5 s = 10 × dt)
MSE_COEF     = 0.1    # poids du terme MSE dans la pred loss
NORM_COEF    = 0.0    # désactivé
SEQ_LEN      = 100    # fenêtre tirée aléatoirement dans chaque trajectoire
N_PROJ       = 512
EPOCHS       = 50
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS  = 2      # Colab has 2 CPU cores
CKPT_DIR     = 'checkpoints'
VIS_DIR      = 'visuals'
RESUME       = None   # set to 'checkpoints/jepa/lewm_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.jepa.model import LeWorldModel
from data.dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = LeWorldModel(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    lam=LAM,
    mse_coef=MSE_COEF,
    norm_coef=NORM_COEF,
    n_proj=N_PROJ,
    ema_momentum=EMA_MOMENTUM,
    rollout_k=ROLLOUT_K,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sums = {'loss': 0.0, 'pred_loss': 0.0, 'sigreg': 0.0}
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True))
        for k in sums:
            sums[k] += m[k].item()
    n = len(loader)
    return {k: v / n for k, v in sums.items()}

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

# ── Monitoring : puissance GPU · mémoire · temps ───────────────────────────────
import threading, subprocess

_power_readings = []
_stop_monitor   = threading.Event()

def _monitor_power():
    while not _stop_monitor.is_set():
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
                text=True
            ).strip()
            _power_readings.append(float(out))
        except Exception:
            pass
        time.sleep(2)

torch.cuda.reset_peak_memory_stats()
_monitor_thread    = threading.Thread(target=_monitor_power, daemon=True)
_monitor_thread.start()
_train_start_wall  = time.time()
print("Monitoring démarré : puissance GPU (toutes les 2s), mémoire peak, temps total.")


In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history             = {k: [] for k in ['train', 'val', 'pred', 'sigreg']}
best_val            = float('inf')
best_epoch          = 1
time_per_epoch      = []
memory_per_epoch_MB = []

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0   = time.time()
    sums = {'loss': 0.0, 'pred_loss': 0.0, 'sigreg': 0.0}

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        model.update_target()
        for k in sums:
            sums[k] += m[k].item()

    scheduler.step()
    n          = len(train_loader)
    train_loss = sums['loss'] / n
    val_m      = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_m['loss'])
    history['pred'].append(sums['pred_loss'] / n)
    history['sigreg'].append(sums['sigreg'] / n)

    elapsed = time.time() - t0
    time_per_epoch.append(round(elapsed, 2))
    memory_per_epoch_MB.append(round(torch.cuda.max_memory_allocated() / 1e6, 1))

    improved = val_m['loss'] < best_val
    if improved:
        best_val   = val_m['loss']
        best_epoch = epoch
        torch.save({
            'epoch':     epoch,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss':  best_val,
            'args': {
                'embed_dim':    EMBED_DIM,
                'hidden_dim':   HIDDEN_DIM,
                'lam':          LAM,
                'mse_coef':     MSE_COEF,
                'norm_coef':    NORM_COEF,
                'n_proj':       N_PROJ,
                'ema_momentum': EMA_MOMENTUM,
                'rollout_k':    ROLLOUT_K,
            },
        }, f'{CKPT_DIR}/lewm_best.pt')

    lr_now = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  loss={train_loss:.4f}'
        f'  pred={sums["pred_loss"]/n:.4f}'
        f'  sig={sums["sigreg"]/n:.4f}'
        f'  val={val_m["loss"]:.4f}'
        f'  lr={lr_now:.2e}'
        f'  mem={memory_per_epoch_MB[-1]:.0f}MB'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

print(f'\nBest val loss: {best_val:.4f}  @epoch {best_epoch}  ->  {CKPT_DIR}/lewm_best.pt')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#111')

# ── Total loss ─────────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val')
ax.set_title('Total loss', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

# ── Pred + SIGReg ──────────────────────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['pred'],   color='#a5d6a7', label='pred loss (cosine+MSE)')
ax.plot(epochs_x, history['sigreg'], color='#ce93d8', label='SIGReg')
ax.set_title('Pred + SIGReg', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Latent space visualisation (PCA)

JEPA ne peut pas décoder en pixels — on visualise à la place la **structure de l'espace latent**.

- Chaque point = embedding z d'une frame
- Couleur = progression temporelle dans la trajectoire (bleu → rouge)
- **Si le modèle a bien appris :** les trajectoires forment des courbes lisses et séparées dans l'espace PCA — signature que z encode θ ET ω
- **Si effondrement :** tous les points sont concentrés, les trajectoires se superposent

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(f'{CKPT_DIR}/lewm_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (val_loss={ckpt["val_loss"]:.4f})')

# ── Encode a few val trajectories ─────────────────────────────────────────────
N_TRAJS  = 8   # nombre de trajectoires à visualiser
all_z    = []
all_time = []
all_traj = []

with torch.no_grad():
    for i, (frames, _) in enumerate(val_loader):
        if i >= N_TRAJS:
            break
        z = model.encode(frames[:1].to(device))   # (1, T, D)
        T = z.shape[1]
        all_z.append(z[0].cpu().numpy())           # (T, D)
        all_time.append(np.linspace(0, 1, T))
        all_traj.append(np.full(T, i))

Z    = np.concatenate(all_z,    axis=0)   # (N_TRAJS*T, D)
time = np.concatenate(all_time, axis=0)
traj = np.concatenate(all_traj, axis=0)

# ── PCA (2D) ───────────────────────────────────────────────────────────────────
Z_centered = Z - Z.mean(0)
_, _, Vt   = np.linalg.svd(Z_centered, full_matrices=False)
PC         = Z_centered @ Vt[:2].T    # (N, 2)

var_explained = np.var(PC, axis=0) / np.var(Z_centered, axis=None)
print(f'PC1 var explained: {var_explained[0]:.1%}  |  PC2: {var_explained[1]:.1%}')

# ── Plot ───────────────────────────────────────────────────────────────────────
cmap   = get_cmap('plasma')
colors = ['#4fc3f7', '#ff8a65', '#a5d6a7', '#ce93d8',
          '#ffcc80', '#ef9a9a', '#80cbc4', '#f48fb1']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111')

# Panel 1 : coloré par trajectoire
ax = axes[0]
ax.set_facecolor('#111')
for i in range(N_TRAJS):
    mask = traj == i
    ax.plot(PC[mask, 0], PC[mask, 1], color=colors[i], alpha=0.7, lw=1.2, label=f'traj {i}')
    ax.scatter(PC[mask, 0][0],  PC[mask, 1][0],  color=colors[i], s=60, marker='o', zorder=5)  # start
    ax.scatter(PC[mask, 0][-1], PC[mask, 1][-1], color=colors[i], s=60, marker='x', zorder=5)  # end
ax.set_title('PCA — trajectoires (o=start, x=end)', color='white')
ax.set_xlabel('PC1', color='white')
ax.set_ylabel('PC2', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444', fontsize=7)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

# Panel 2 : coloré par progression temporelle (toutes traj confondues)
ax = axes[1]
ax.set_facecolor('#111')
sc = ax.scatter(PC[:, 0], PC[:, 1], c=time, cmap='plasma', s=4, alpha=0.6)
cbar = plt.colorbar(sc, ax=ax)
cbar.ax.yaxis.set_tick_params(color='white')
cbar.set_label('t/T', color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')
ax.set_title('PCA — progression temporelle', color='white')
ax.set_xlabel('PC1', color='white')
ax.set_ylabel('PC2', color='white')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_latent_pca.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Section 8 — Post-training analysis                                          ║
# ║  Énergie · Mémoire · Temps · Probe linéaire + MLP · Rollout quality latent   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import json
from datetime import datetime
from data.dataset import PendulumSeqDataset
from torch.utils.data import random_split
import torch.nn.functional as F

# ── Arrêt monitor + totaux ─────────────────────────────────────────────────────
_stop_monitor.set()
_total_time_s = time.time() - _train_start_wall
avg_power_W   = float(np.mean(_power_readings)) if _power_readings else 0.0
energy_Wh     = avg_power_W * _total_time_s / 3600
peak_mem_MB   = torch.cuda.max_memory_allocated() / 1e6
n_params      = sum(p.numel() for p in model.parameters())
n_grad_steps  = len(history['train']) * len(train_loader)

print(f"Temps total    : {_total_time_s/60:.1f} min  ({n_grad_steps:,} gradient steps)")
print(f"Énergie GPU    : {avg_power_W:.1f} W moy  →  {energy_Wh:.3f} Wh  ({avg_power_W*_total_time_s:.0f} J)")
print(f"Mémoire pic    : {peak_mem_MB:.0f} MB  |  Paramètres : {n_params/1e6:.3f} M")

# ── Reload best checkpoint ─────────────────────────────────────────────────────
ckpt_best = torch.load(f'{CKPT_DIR}/lewm_best.pt', map_location=device)
model.load_state_dict(ckpt_best['model'])
model.eval()

# ── Helpers probe ──────────────────────────────────────────────────────────────
def _extract_z(mdl, indices, ds_dir, dev, chunk=16):
    full_ds = PendulumSeqDataset(ds_dir, seq_len=None)
    Zs, Ls  = [], []
    for idx in indices:
        frames, states = full_ds[idx]
        T  = frames.shape[0]
        zc = []
        for s in range(0, T, chunk):
            f = frames[s:s+chunk].unsqueeze(0).to(dev)
            with torch.no_grad():
                zc.append(mdl.encode(f)[0].cpu())
        z_all = torch.cat(zc, 0)
        Zs.append(z_all[1:])
        Ls.append(states[1:])
    return torch.cat(Zs).float(), torch.cat(Ls).float()

def _r2(pred, true):
    ss_res = ((true - pred)**2).sum(0)
    ss_tot = ((true - true.mean(0))**2).sum(0)
    r = 1 - ss_res / ss_tot.clamp(min=1e-8)
    return {'r2_theta': round(r[0].item(), 4), 'r2_omega': round(r[1].item(), 4),
            'r2_mean': round(r.mean().item(), 4)}

def _run_probe(Z_tr, L_tr, Z_val, L_val, epochs=150, hidden=256):
    import torch.nn as nn, torch.utils.data as td
    zm, zs = Z_tr.mean(0), Z_tr.std(0).clamp(min=1e-6)
    Zn_tr  = (Z_tr - zm) / zs
    Zn_val = (Z_val - zm) / zs
    D, out = Zn_tr.shape[1], {}
    for name, net, lr in [
        ('linear', nn.Linear(D, 2), 1e-2),
        ('mlp',    nn.Sequential(nn.Linear(D, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 2)), 1e-3),
    ]:
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        dl  = td.DataLoader(td.TensorDataset(Zn_tr, L_tr), batch_size=1024, shuffle=True)
        for _ in range(epochs):
            net.train()
            for zb, lb in dl:
                loss = F.mse_loss(net(zb), lb)
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            out[name] = _r2(net(Zn_val), L_val)
        print(f"  probe_{name:6s}  R²(θ)={out[name]['r2_theta']:.4f}  "
              f"R²(ω)={out[name]['r2_omega']:.4f}  R²(mean)={out[name]['r2_mean']:.4f}")
    return out

# ── Rollout quality — espace latent ───────────────────────────────────────────
@torch.no_grad()
def _rollout_latent(mdl, val_idx, ds_dir, dev, steps=(1, 5, 10, 20), n_traj=30):
    full_ds = PendulumSeqDataset(ds_dir, seq_len=None)
    cos_acc = {k: [] for k in steps}
    mse_acc = {k: [] for k in steps}
    max_k   = max(steps)
    for idx in list(val_idx)[:n_traj]:
        frames, _ = full_ds[idx]
        T = frames.shape[0]
        if T < max_k + 5: continue
        z = mdl.encode(frames.unsqueeze(0).to(dev))[0]    # (T, D)
        for t in range(1, min(T - max_k, 50)):
            z_r = z[t:t+1]
            for k in range(1, max_k + 1):
                z_r = mdl.predictor(z_r)
                if k in steps:
                    z_true = z[t+k:t+k+1]
                    cos_acc[k].append(F.cosine_similarity(z_r, z_true).item())
                    mse_acc[k].append(F.mse_loss(z_r, z_true).item())
    return {
        **{f'cosine_sim_step{k}': round(float(np.mean(cos_acc[k])), 4)
           for k in steps if cos_acc[k]},
        **{f'latent_mse_step{k}': round(float(np.mean(mse_acc[k])), 6)
           for k in steps if mse_acc[k]},
    }

# ── Run ────────────────────────────────────────────────────────────────────────
print('\n[1/2] Probe linéaire + MLP…')
full_ds = PendulumSeqDataset(DATASET_DIR, seq_len=None)
n_ds    = len(full_ds)
gen     = torch.Generator().manual_seed(42)
tr_idx, val_idx = random_split(range(n_ds), [int(n_ds*0.9), n_ds - int(n_ds*0.9)], generator=gen)
Z_tr, L_tr   = _extract_z(model, list(tr_idx),  DATASET_DIR, device)
Z_val, L_val = _extract_z(model, list(val_idx), DATASET_DIR, device)
probe_res    = _run_probe(Z_tr, L_tr, Z_val, L_val)

print('\n[2/2] Rollout quality latent…')
rollout_q = _rollout_latent(model, list(val_idx), DATASET_DIR, device)
for k, v in rollout_q.items(): print(f'  {k}: {v}')

# ── Save JSON ──────────────────────────────────────────────────────────────────
stats = {
    'model':     'jepa',
    'timestamp': datetime.now().isoformat(),
    'hyperparams': {
        'embed_dim': EMBED_DIM, 'hidden_dim': HIDDEN_DIM,
        'lam': LAM, 'rollout_k': ROLLOUT_K, 'mse_coef': MSE_COEF,
        'norm_coef': NORM_COEF, 'n_proj': N_PROJ,
        'ema_momentum': EMA_MOMENTUM, 'seq_len': SEQ_LEN,
        'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
        'lr': LR, 'weight_decay': WEIGHT_DECAY,
    },
    'time': {
        'total_s':         round(_total_time_s, 1),
        'total_min':       round(_total_time_s / 60, 2),
        'per_epoch_s':     time_per_epoch,
        'avg_per_epoch_s': round(float(np.mean(time_per_epoch)), 2),
        'gradient_steps':  n_grad_steps,
        'steps_per_sec':   round(n_grad_steps / _total_time_s, 2),
    },
    'memory': {
        'peak_gpu_MB':       round(peak_mem_MB, 1),
        'model_params_M':    round(n_params / 1e6, 3),
        'trainable_params':  n_params,
        'per_epoch_peak_MB': memory_per_epoch_MB,
        'max_per_epoch_MB':  max(memory_per_epoch_MB),
    },
    'energy': {
        'avg_power_W':      round(avg_power_W, 2),
        'total_Wh':         round(energy_Wh, 4),
        'total_J':          round(avg_power_W * _total_time_s, 1),
        'n_power_samples':  len(_power_readings),
        'power_readings_W': [round(p, 1) for p in _power_readings],
    },
    'convergence': {
        'best_val_loss':     round(best_val, 6),
        'best_epoch':        best_epoch,
        'epochs_trained':    len(history['train']),
        'train_loss':        [round(x, 6) for x in history['train']],
        'val_loss':          [round(x, 6) for x in history['val']],
        'pred_loss':         [round(x, 6) for x in history['pred']],
        'sigreg':            [round(x, 6) for x in history['sigreg']],
    },
    'probe_linear':   probe_res['linear'],
    'probe_mlp':      probe_res['mlp'],
    'rollout_latent': rollout_q,
}

out_path = f'{VIS_DIR}/training_stats_jepa.json'
Path(VIS_DIR).mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\n{'═'*58}")
print(f"  Stats → {out_path}")
print(f"  Temps    : {_total_time_s/60:.1f} min  |  {n_grad_steps:,} gradient steps")
print(f"  Énergie  : {energy_Wh:.3f} Wh  ({avg_power_W*_total_time_s:.0f} J)")
print(f"  Mémoire  : {peak_mem_MB:.0f} MB pic")
print(f"  Probe    : linear R²={probe_res['linear']['r2_mean']:.4f}  "
      f"mlp R²={probe_res['mlp']['r2_mean']:.4f}")
print(f"{'═'*58}")

## 8. Download checkpoint

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/lewm_best.pt')